In [26]:
#----IMPORT LIBRAIRIES----
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

import pvlib

import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

import Model_func as mf
import boto3

from dotenv import load_dotenv
import os

load_dotenv()

True

In [27]:
#---VARIABLES----
weather_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/openweathermap/merge_openweathermap_cleaned.csv'
solar_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/solar/raw_solar_data.csv'
landsat_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/LandSat/result_EarthExplorer_region_ARA.csv'

prod_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/prod/eCO2mix_RTE_Auvergne-Rhone-Alpes_cleaned.csv'
target = 'tch_solaire_(%)'


In [28]:
# #--- PREPARATION ----
# #---data_prep : collect and merge
# df = mf.data_prep(weather_data_path, solar_data_path, landsat_data_path)
# df_copy = df.copy()

# #---data_split : add target and train_test_split
# prod_data = mf.data_collection_prod(prod_data_path)
# data = mf.add_target(df_copy, prod_data, target_columns_to_use=['Time', target])


In [29]:

collected_weather_data = mf.data_collection_weather(weather_data_path) # collect data and format columns per city
collected_solar_data = mf.data_coll_solar(solar_data_path)
collected_landsat_data = mf.data_coll_landsat(landsat_data_path)
landsat_data = collected_landsat_data.copy()

weather_solar = mf.merge_weather_solar_data(collected_weather_data, collected_solar_data)

#creer un df landsat réduit avec 1 donnée/jour
columns_to_keep = landsat_data.select_dtypes(exclude=["object"]).columns
limited_landsat_data = landsat_data[columns_to_keep].groupby('Time').mean().reset_index()
 
merged_data = mf.merge_weather_solar_landsat_data(collected_weather_data, collected_solar_data, limited_landsat_data)

In [30]:
#---data_split : add target and train_test_split
prod_data = mf.data_collection_prod(prod_data_path)
data = mf.add_target(merged_data, prod_data, target_columns_to_use=['Time', target])
data = data.dropna(axis=1)

y = data[target].to_numpy()
X = data.drop(target, axis=1)
x_train, x_test, y_train, y_test = train_test_split(X, y)


In [47]:

#---MLFlow params
os.environ["APP_URI"] = "https://renergies99-mlflow.hf.space/"
EXPERIMENT_NAME = "all_columns_models"

mlflow.set_tracking_uri(os.environ["APP_URI"])
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

mlflow.sklearn.autolog()  # enables automatic logging for scikit-learn

#---pipeline prep
numeric_cols = X.select_dtypes(include='number').columns.tolist()
object_cols = X.select_dtypes(exclude='number').columns.tolist()

transformers = [('num', StandardScaler(), numeric_cols)]
# if object_cols:  
#     transformers.append(('obj', 'passthrough', object_cols))

preprocessor = ColumnTransformer(transformers=transformers)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('estimator', LinearRegression())
])

run_description = f"Features used: all\nTarget: {target}"

with mlflow.start_run(experiment_id=experiment.experiment_id, description=run_description):
    # Fit the pipeline (preprocessing + model)
    pipeline.fit(x_train, y_train)

    # predictions
    y_pred = pipeline.predict(x_test)

    # metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    n = len(y_test)
    p = X.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    
    # logging metrics
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.log_metric("Adjusted_R2", adj_r2)
 
    # Log the full pipeline as a model
    mlflow.sklearn.log_model(pipeline, artifact_path="pipeline_model")

# initiate model with:
# model_full = ServingModel(model, preprocessor)

# Call to log model on mlflow
# mlflow.pyfunc.log_model("model", python_model=AutoEncoderServingModel(model, preprocessing_transform))



# class ServingModel(mlflow.pyfunc.PythonModel):
#     def __init__(self, model , preprocessing_transform):
#         self._model = model
#         self._preprocessing_transform = preprocessing_transform

#     def predict(self, model_input):
#         """
#         Perform a transformation and predict on input of (batch, sequence, features)
#         """
#         for i in range(model_input.shape[0]):
#             model_input[i, :] = self._preprocessing_transform(model_input[i, :])

#         return self._model.predict(model_input)

2025/11/19 16:18:47 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\hardy\anaconda3\envs\Jedi\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/11/19 16:18:49 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\hardy\anaconda3\envs\Jedi\Lib\site-p

🏃 View run crawling-gull-905 at: https://renergies99-mlflow.hf.space/#/experiments/5/runs/5af5104e94fe40d2948ca5471e2e7d72
🧪 View experiment at: https://renergies99-mlflow.hf.space/#/experiments/5


In [43]:
numeric_cols

['Moulins_sunrise',
 'Moulins_sunset',
 'Moulins_temp',
 'Moulins_feels_like',
 'Moulins_pressure',
 'Moulins_humidity',
 'Moulins_dew_point',
 'Moulins_clouds',
 'Moulins_wind_speed',
 'Moulins_wind_deg',
 'Moulins_lat',
 'Moulins_lon',
 'Time',
 'Moulins_Month',
 'Moulins_apparent_zenith',
 'Moulins_zenith',
 'Moulins_apparent_elevation',
 'Moulins_elevation',
 'Moulins_azimuth',
 'Moulins_equation_of_time',
 'Moulins_day_length',
 'Aurillac_sunrise',
 'Aurillac_sunset',
 'Aurillac_temp',
 'Aurillac_feels_like',
 'Aurillac_pressure',
 'Aurillac_humidity',
 'Aurillac_dew_point',
 'Aurillac_clouds',
 'Aurillac_wind_speed',
 'Aurillac_wind_deg',
 'Aurillac_lat',
 'Aurillac_lon',
 'Aurillac_Month',
 'Aurillac_apparent_zenith',
 'Aurillac_zenith',
 'Aurillac_apparent_elevation',
 'Aurillac_elevation',
 'Aurillac_azimuth',
 'Aurillac_equation_of_time',
 'Aurillac_day_length',
 'Saint-Étienne_sunrise',
 'Saint-Étienne_sunset',
 'Saint-Étienne_temp',
 'Saint-Étienne_feels_like',
 'Saint-Étie

In [45]:
data[numeric_cols]

,Moulins_sunrise,Moulins_sunset,Moulins_temp,Moulins_feels_like,Moulins_pressure,Moulins_humidity,Moulins_dew_point,Moulins_clouds,Moulins_wind_speed,Moulins_wind_deg,...,Nyons_zenith,Nyons_apparent_elevation,Nyons_elevation,Nyons_azimuth,Nyons_equation_of_time,Nyons_day_length,nb_event,SSN,K index Boulder,K index Planetary
0,2021-01-09 07:29:19,2021-01-09 16:17:54,1.11,-2.22,1020,83,-1.28,100,3.06,38,...,67.249193,22.790471,22.750807,168.282831,-7.183662,9.065833,1,0,0.500,0.125
1,2021-01-10 07:28:56,2021-01-10 16:19:05,0.71,-2.78,1022,73,-3.16,37,3.16,21,...,67.117041,22.922374,22.882959,168.156344,-7.590525,9.089722,1,0,0.625,0.375
2,2021-01-11 07:28:31,2021-01-11 16:20:18,-0.74,-2.53,1027,79,-3.56,70,1.46,291,...,66.977711,23.061445,23.022289,168.030511,-7.988016,9.114722,1,0,1.875,2.125
3,2021-01-12 07:28:03,2021-01-12 16:21:32,4.55,0.86,1023,92,3.36,100,4.83,250,...,66.831258,23.207629,23.168742,167.905434,-8.375673,9.140278,1,0,2.000,1.750
4,2021-01-13 07:27:32,2021-01-13 16:22:48,8.50,6.07,1025,97,8.05,100,4.19,278,...,66.677737,23.360871,23.322263,167.781216,-8.753044,9.167500,1,0,1.125,0.375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1707,2025-09-25 05:37:22,2025-09-25 17:39:17,11.47,11.00,1021,89,9.72,94,1.60,315,...,49.721046,40.298779,40.278954,149.526826,8.380755,12.035833,4,149,1.875,1.875
1708,2025-09-26 05:38:39,2025-09-26 17:37:17,11.02,10.55,1020,91,9.61,91,0.36,176,...,50.053344,39.966713,39.946656,149.815229,8.723571,11.985000,1,137,1.875,0.000
1709,2025-09-27 05:39:56,2025-09-27 17:35:17,14.34,14.13,1021,88,12.38,82,1.68,145,...,50.386489,39.633806,39.613511,150.099809,9.063336,11.934167,8,172,1.625,0.000
1710,2025-09-28 05:41:14,2025-09-28 17:33:17,13.84,13.47,1020,84,11.19,95,1.54,100,...,50.720394,39.300142,39.279606,150.380485,9.399736,11.883333,10,131,2.125,0.000
